# Does There Reply? - Local Mistral Zero-Shot Classifier

This notebook uses a local Mistral model through Ollama. It predicts binary labels `0/1` without training on labels, evaluates on `validation.csv`, and creates a test submission.

In [1]:
from pathlib import Path
import json
import re
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, mean_absolute_error

DATA_DIR = Path.cwd()
if not (DATA_DIR / 'train.csv').exists():
    DATA_DIR = DATA_DIR / 'doesthereply'

# Experiment settings.
OLLAMA_URL = 'http://localhost:11434/api/generate'
OLLAMA_MODEL = 'mistral'
TEMPERATURE = 0.0
NUM_PREDICT = 3
REQUEST_TIMEOUT_SECONDS = 120

train_df = pd.read_csv(DATA_DIR / 'train.csv')
dev_df = pd.read_csv(DATA_DIR / 'validation.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
print(f'Train: {len(train_df)} | validation: {len(dev_df)} | test: {len(test_df)}')

Train: 12 | validation: 4 | test: 8


## Zero-shot Mistral prompt

Mistral receives only the question text and must return one integer: `1` if it appears answerable with a useful reply, or `0` otherwise.

In [2]:
SYSTEM_PROMPT = 'Classify whether this home-improvement question is likely to receive a useful reply. Return exactly one integer from 0 to 1 and nothing else. Return 1 if it is a clear, answerable question; return 0 if it is unlikely to receive a useful reply.'

def build_prompt(text):
    return f'{SYSTEM_PROMPT}\n\nQuestion text:\n{str(text).strip()}\n\nLabel (0 or 1):'

print(build_prompt('TITLE\nHow can I repair a leaking pipe?\n\nQUESTION\nThe pipe leaks near the joint.'))

Classify whether this home-improvement question is likely to receive a useful reply. Return exactly one integer from 0 to 1 and nothing else. Return 1 if it is a clear, answerable question; return 0 if it is unlikely to receive a useful reply.

Question text:
TITLE
How can I repair a leaking pipe?

QUESTION
The pipe leaks near the joint.

Label (0 or 1):


In [3]:
def parse_label(response_text):
    match = re.fullmatch(r'\s*\(?([01])\)?(?:\s*[().,:-])?\s*', response_text)
    if match is None:
        raise ValueError(f'Mistral returned a non-label response: {response_text!r}')
    return int(match.group(1))


def mistral_score(text):
    payload = {
        'model': OLLAMA_MODEL, 'prompt': build_prompt(text), 'stream': False,
        'options': {'temperature': TEMPERATURE, 'num_predict': NUM_PREDICT},
    }
    request = Request(OLLAMA_URL, data=json.dumps(payload).encode('utf-8'), headers={'Content-Type': 'application/json'}, method='POST')
    try:
        with urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as response:
            result = json.loads(response.read().decode('utf-8'))
    except HTTPError as exc:
        body = exc.read().decode('utf-8', errors='replace')
        raise RuntimeError(f'Ollama HTTP {exc.code}: {body}') from exc
    except URLError as exc:
        raise RuntimeError('Could not connect to Ollama. Run: ollama pull mistral') from exc
    if 'response' not in result:
        raise RuntimeError(f'Ollama response did not contain text: {result}')
    return parse_label(result['response'])


def predict_frame(frame):
    predictions = []
    for row_number, text in enumerate(frame['text'], start=1):
        predictions.append(mistral_score(text))
        if row_number % 5 == 0 or row_number == len(frame):
            print(f'Predicted {row_number}/{len(frame)} rows')
    return np.asarray(predictions, dtype=int)

print(f'Configured local model: {OLLAMA_MODEL}')

Configured local model: mistral


In [4]:
dev_predictions = predict_frame(dev_df)
dev_actual = dev_df['label'].to_numpy(dtype=int)
print(f'Validation MAE: {mean_absolute_error(dev_actual, dev_predictions):.4f}')
print(f'Validation macro-F1: {f1_score(dev_actual, dev_predictions, average="macro", zero_division=0):.4f}')

Predicted 4/4 rows
Validation MAE: 0.2500
Validation macro-F1: 0.4286


In [5]:
test_predictions = predict_frame(test_df)
submission = pd.DataFrame({'id': test_df['id'], 'label': test_predictions})
submission_path = DATA_DIR / 'submission_mistral_zero_shot.csv'
submission.to_csv(submission_path, index=False)
print(f'Saved {len(submission)} predictions to {submission_path}')
if 'label' in test_df.columns:
    print(f'Test macro-F1: {f1_score(test_df["label"], test_predictions, average="macro", zero_division=0):.4f}')
else:
    print('Test macro-F1 unavailable: test.csv has no ground-truth label column.')
submission.head()

Predicted 5/8 rows
Predicted 8/8 rows
Saved 8 predictions to c:\Users\bradm\Documents\GitHub\ai-masters-workspace\ARI-510-Machine-Learning\Phase 2b\doesthereply\submission_mistral_zero_shot.csv
Test macro-F1 unavailable: test.csv has no ground-truth label column.


,id,label
0,DIY_328278_328285,1
1,DIY_328287_328288,1
2,DIY_328293_328297,1
3,DIY_328296_328299,1
4,DIY_328327_328330,1
